# Power iteration and normalization

Run the CPU pilot first. This notebook runs the same Python files as VS Code. Change the Git ref to a commit SHA when you want to reproduce a specific version.

In [ ]:
from pathlib import Path
import subprocess
import sys

repo = Path('/content/power-iteration-normalization')
url = 'https://github.com/tianrumao/power-iteration-normalization.git'
ref = 'main'

def run(*args):
    subprocess.run(args, cwd=repo, check=True)

if not repo.exists():
    subprocess.run(['git', 'clone', url, str(repo)], check=True)
run('git', 'fetch', 'origin')
revision = 'origin/main' if ref == 'main' else ref
run('git', 'checkout', '--detach', revision)
run('git', 'rev-parse', 'HEAD')
run(sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt')

In [ ]:
run(sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v')

In [ ]:
from datetime import datetime, timezone
import torch

device = 'cpu'
if device == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before using CUDA.')
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
output = repo / 'runs' / ('colab-pilot-' + stamp)
try:
    run(sys.executable, '-m', 'src.train', '--device', device, '--output', str(output))
finally:
    print('Run folder:', output)

In [ ]:
from IPython.display import Image, display

run(sys.executable, '-m', 'src.evaluate', str(output), '--device', device)
display(Image(filename=str(output / 'layer_profiles.png')))

### Optional — save this run to Drive

Run this cell after training, including if a condition failed. Colab will ask you to authorize the Drive mount.

In [ ]:
from google.colab import drive
from shutil import copytree

drive.mount('/content/drive')
destination = Path('/content/drive/MyDrive/power_iteration/runs') / output.name
destination.parent.mkdir(parents=True, exist_ok=True)
copytree(output, destination)
print(destination)